# ⚙️ 02. Single-Cell Preprocessing & Barcode Error Correction
### Trimming, Poly-G / Poly-A Filtering, and 1-Hamming Distance Error Correction

#### Pipeline Steps:
1. **Quality & Adapter Trimming with `fastp`**: Sliding window Phred filtering, removal of NextSeq/NovaSeq Poly-G dark cycle artifacts, and TSO adapter removal.
2. **Whitelist Error Correction**: 1-Hamming distance error-correction of 1-bp sequencing mutations in cell barcodes.
3. **Clean FASTQ Generation**: Outputting standard paired FASTQ and extracted FASTQ format (`@READ_CB_UMI`).


In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import gzip
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_config
from src.sc_preprocess import SingleCellPreprocessor

config = load_config("../config/pipeline_config.yaml")
print(f"Loaded configuration for sample: {config['project']['sample_id']}")


## 1. Execute `fastp` Quality & Adapter Trimming
We run `fastp` with single-cell parameters:
- Preserve R1 (28bp CB+UMI)
- Trim low-quality bases from R2 3' end
- Strip poly-G and poly-A tails


In [ ]:
preprocessor = SingleCellPreprocessor(config)

temp_r1 = Path("../data/temp/nb_trimmed_R1.fastq.gz")
temp_r2 = Path("../data/temp/nb_trimmed_R2.fastq.gz")
fp_html = Path("../reports/fastp/nb_fastp.html")
fp_json = Path("../reports/fastp/nb_fastp.json")

fp_stats = preprocessor.run_fastp_trimming(
    r1_in=f"../{config['paths']['sample_r1']}",
    r2_in=f"../{config['paths']['sample_r2']}",
    r1_out=temp_r1,
    r2_out=temp_r2,
    report_html=fp_html,
    report_json=fp_json
)

print(f"Fastp filtering completed!")
print(f"Reads before: {fp_stats['summary']['before_filtering']['total_reads']:,}")
print(f"Reads after:  {fp_stats['summary']['after_filtering']['total_reads']:,}")
print(f"Q30 rate before: {fp_stats['summary']['before_filtering']['q30_rate']*100:.2f}%")
print(f"Q30 rate after:  {fp_stats['summary']['after_filtering']['q30_rate']*100:.2f}%")


## 2. Poly-X & Adapter Trimming Results
Let's inspect the exact number of reads trimmed for Poly-A / Poly-G artifacts.


In [ ]:
polyx_info = fp_stats.get("polyx_trimming", {})
polyx_reads = polyx_info.get("polyx_trimmed_reads", {})

df_polyx = pd.DataFrame(list(polyx_reads.items()), columns=["Base", "Trimmed Reads"]).sort_values("Trimmed Reads", ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(data=df_polyx, x="Base", y="Trimmed Reads", palette="viridis")
plt.title("Reads Trimmed by Base Type (Poly-X Filter)", fontweight='bold')
plt.ylabel("Number of Reads")
plt.tight_layout()
plt.show()


## 3. Barcode Error Correction (1-Hamming Distance)
We now map each cell barcode against the 10x whitelist:
- If exact match $\rightarrow$ Keep as-is.
- If 1-bp mismatch with unambiguous whitelist barcode $\rightarrow$ Error-correct to true barcode.
- If $>1$-bp mismatch or ambiguous $\rightarrow$ Discard.


In [ ]:
clean_r1 = Path(f"../{config['paths']['clean_r1']}")
clean_r2 = Path(f"../{config['paths']['clean_r2']}")
extracted_out = Path(f"../{config['paths']['extracted_fastq']}")

bc_stats = preprocessor.filter_and_correct_barcodes(
    r1_in=temp_r1,
    r2_in=temp_r2,
    r1_out=clean_r1,
    r2_out=clean_r2,
    extracted_out=extracted_out,
    whitelist_file=f"../{config['paths']['whitelist_file']}"
)

# Clean up temporary intermediate file
if temp_r1.exists(): temp_r1.unlink()
if temp_r2.exists(): temp_r2.unlink()

pd.DataFrame([bc_stats]).T.rename(columns={0: "Count / Rate"})


## 4. Preprocessing Read Yield Breakdown

In [ ]:
categories = ["Exact Whitelist", "1-bp Error-Corrected", "Discarded Invalid"]
vals = [
    bc_stats["exact_whitelist_matches"],
    bc_stats["corrected_1bp_mismatches"],
    bc_stats["discarded_invalid_barcodes"]
]

plt.figure(figsize=(8, 4.5))
bars = plt.bar(categories, vals, color=["#2e7d32", "#1976d2", "#d32f2f"], width=0.55)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + max(vals)*0.02, f"{yval:,} ({yval/bc_stats['input_reads']*100:.1f}%)", ha='center', fontweight='bold')

plt.title("Barcode Recovery & Filtering Summary", fontweight='bold')
plt.ylabel("Read Count")
plt.ylim(0, max(vals) * 1.15)
plt.tight_layout()
plt.show()
